# Experiment 2: local misspecification and robust weighting

The design fixes $a=2$, $c=1$, and $\beta=1$. We use five sample sizes $n\in\{200,600,1000,1400,1800\}$, three nonzero local-deviation regimes, and the Section-4.1 ellipsoid $\{\tau=Du:\|u\|_2\leq1\}$. Both-channel deviation is evaluated at 16 boundary directions. The unadjusted OLS baseline omits the confounder and is shown for MSE comparison. Its fixed bias lies outside the local moment-deviation model, so no Remark-3 robust interval or theoretical local risk is assigned to OLS.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import ndtr
from IPython.display import display

root = Path.cwd()
if not (root / "results" / "local_misspec" / "local_misspec_worst_case.csv").exists():
    root = root / "unification-control-modelling"
result_dir = root / "results" / "local_misspec"
FIGURE_DIR = root / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def save_figure(fig, stem):
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", dpi=300, bbox_inches="tight",
                pad_inches=0.22, facecolor="white")

corners = pd.read_csv(result_dir / "local_misspec_corners.csv")
worst = pd.read_csv(result_dir / "local_misspec_worst_case.csv")
assert set(corners["replications"]) == {1000}
assert set(worst["kappa"]) == {0.0, 0.5, 1.0}
assert set(worst["n"]) == {200, 600, 1000, 1400, 1800}
assert set(worst["method"]) == {"ols_unadjusted", "iv", "sv", "equal", "local_risk"}
assert np.allclose(corners["scaled_mse"], corners["scaled_squared_bias"] + corners["scaled_variance"], atol=1e-8)

regimes = ("iv", "sv", "both")
regime_labels = {"iv": "IV channel may deviate", "sv": "SV channel may deviate", "both": "Both channels may deviate"}
methods = ("ols_unadjusted", "iv", "sv", "equal", "local_risk")
robust_methods = methods[1:]
inference_methods = methods
method_labels = {"ols_unadjusted": "Unadjusted OLS", "iv": "IV only", "sv": "SV only", "equal": "Equal GMM", "local_risk": "Local-risk GMM"}
palette = sns.color_palette("colorblind", 8)
colors = dict(zip(methods, (palette[i] for i in (7, 0, 1, 2, 4))))
markers = {"ols_unadjusted": "X", "iv": "o", "sv": "s", "equal": "^", "local_risk": "D"}
n_values = sorted(worst["n"].unique())
print(f"Loaded {len(corners)} boundary summaries and {len(worst)} worst-case summaries.")


Loaded 1025 boundary summaries and 175 worst-case summaries.


## Both channels may deviate: MSE and inference across sample sizes

At $\kappa=1$, the three panels separately show worst-case MSE, Type I error, and minimum local power at all five sample sizes. Each panel has a method legend below the axis. OLS uses Wald inference; IV, SV, and GMM use local-robust inference.


In [2]:
regime, kappa = "both", 1.0
panel = worst[worst["regime"].eq(regime) & worst["kappa"].eq(kappa)].copy()
panel["type_i_error"] = np.where(
    panel["method"].eq("ols_unadjusted"),
    1 - panel["min_wald_coverage"],
    1 - panel["min_robust_coverage"],
)

# Invert the same Wald/robust intervals at each n under beta = 1 + 2/sqrt(n).
raw_columns = ["regime", "kappa", "n", "tau_z", "tau_w",
               "method", "beta_hat", "sigma", "bias_bound"]
raw = pd.read_csv(result_dir / "local_misspec_replications.csv", usecols=raw_columns)
raw = raw[raw["regime"].eq(regime) & raw["kappa"].eq(kappa) &
          raw["method"].isin(inference_methods)].copy()
t = np.where(raw["method"].eq("ols_unadjusted"), 0.0,
             (raw["bias_bound"] / raw["sigma"]).to_numpy())
lower = np.zeros_like(t)
upper = t + 1.959963984540054
for _ in range(45):
    midpoint = (lower + upper) / 2
    below_target = ndtr(midpoint - t) - ndtr(-midpoint - t) < 0.95
    lower = np.where(below_target, midpoint, lower)
    upper = np.where(below_target, upper, midpoint)
critical = (lower + upper) / 2
null_z = np.sqrt(raw["n"].to_numpy()) * (raw["beta_hat"].to_numpy() - 1.0) / raw["sigma"].to_numpy()
raw["local_power"] = np.abs(null_z + 2.0 / raw["sigma"].to_numpy()) > critical
boundary_power = raw.groupby(["n", "tau_z", "tau_w", "method"], as_index=False)["local_power"].mean()
min_power = boundary_power.groupby(["n", "method"], as_index=False)["local_power"].min()

from matplotlib.patches import Patch

journal_style = {
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix", "pdf.fonttype": 42,
    "font.size": 20, "axes.linewidth": 0.7, "axes.edgecolor": "0.65",
    "axes.labelsize": 21, "axes.titlesize": 22,
    "xtick.labelsize": 18, "ytick.labelsize": 18,
    "legend.fontsize": 18, "legend.title_fontsize": 19,
}
with plt.rc_context(journal_style):
    fig, axes = plt.subplots(1, 3, figsize=(18.5, 7.2))
    for method in methods:
        frame = panel[panel["method"].eq(method)].sort_values("n")
        axes[0].plot(frame["n"], frame["worst_mse"],
                     color=colors[method], marker=markers[method], linewidth=1.8,
                     markersize=6, markerfacecolor="white", markeredgewidth=1.2,
                     label=method_labels[method])
    x = np.arange(len(n_values))
    bar_width = 0.14
    type_i_values = panel.pivot(index="n", columns="method", values="type_i_error").reindex(
        index=n_values, columns=list(methods)
    )
    power_values = min_power.pivot(index="n", columns="method", values="local_power").reindex(
        index=n_values, columns=list(methods)
    )
    for axis, values in ((axes[1], type_i_values), (axes[2], power_values)):
        for index, method in enumerate(methods):
            axis.bar(x + (index - 2) * bar_width, values[method], bar_width,
                     color=colors[method], edgecolor="none", linewidth=0)
    axes[0].set(title="(a) Worst-case mean squared error", ylabel="MSE", yscale="log")
    axes[1].set(title="(b) Type I error", ylabel="Worst-case rejection probability",
                ylim=(0, 1.04))
    axes[2].set(title=r"(c) Minimum local power: $\beta=1+2/\sqrt{n}$",
                ylabel="Minimum rejection probability", ylim=(0, 1.04))
    axes[1].axhline(0.05, color="0.45", linestyle=":", linewidth=0.8)
    for ax in axes:
        ax.set_xlabel(r"Sample size $n$")
        ax.grid(axis="y", color="0.88", linewidth=0.5)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(direction="out", length=3, width=0.7)
    axes[0].set_xticks(n_values)
    for ax in axes[1:]:
        ax.set_xticks(x, n_values)
    line_handles, line_labels = axes[0].get_legend_handles_labels()
    axes[0].legend(handles=line_handles, labels=line_labels, title="Method",
                   loc="upper center", bbox_to_anchor=(0.5, -0.17),
                   ncol=2, frameon=False, columnspacing=0.8, handlelength=1.7)
    method_handles = [
        Patch(facecolor=colors[method], edgecolor="none", linewidth=0,
              label=method_labels[method]) for method in methods
    ]
    for ax in axes[1:]:
        ax.legend(handles=method_handles, title="Method", loc="upper center",
                  bbox_to_anchor=(0.5, -0.17), ncol=2, frameon=False,
                  columnspacing=0.8, handlelength=1.3)
    fig.text(0.5, -0.025, "OLS uses Wald inference; IV, SV and GMM use local-robust inference.",
             ha="center", fontsize=17)
    fig.subplots_adjust(left=0.065, right=0.99, bottom=0.35, top=0.92, wspace=0.30)
    save_figure(fig, "local_misspec_mse_comparison")
    display(fig)
    plt.close(fig)
display(panel.pivot(index="n", columns="method", values="worst_mse")[list(methods)].rename(columns=method_labels).round(5))
print("Type I error (OLS Wald / others robust)")
display(panel.pivot(index="n", columns="method", values="type_i_error")[list(methods)].rename(columns=method_labels).round(3))
print("Minimum local power (OLS Wald / others robust)")
display(min_power.pivot(index="n", columns="method", values="local_power")[list(methods)].rename(columns=method_labels).round(3))
del raw


<Figure size 1850x720 with 3 Axes>

method,Unadjusted OLS,IV only,SV only,Equal GMM,Local-risk GMM
n,,,,,
200,0.03485,0.00702,0.00990,0.00453,0.00392
600,0.03164,0.00210,0.00352,0.00146,0.00130
1000,0.03067,0.00130,0.00202,0.00087,0.00074
1400,0.03020,0.00090,0.00150,0.00062,0.00055
1800,0.02969,0.00066,0.00114,0.00047,0.00041


Type I error (OLS Wald / others robust)


method,Unadjusted OLS,IV only,SV only,Equal GMM,Local-risk GMM
n,,,,,
200,0.995,0.067,0.050,0.064,0.059
600,1.000,0.060,0.059,0.055,0.054
1000,1.000,0.059,0.050,0.053,0.048
1400,1.000,0.055,0.061,0.056,0.060
1800,1.000,0.053,0.060,0.051,0.050


Minimum local power (OLS Wald / others robust)


method,Unadjusted OLS,IV only,SV only,Equal GMM,Local-risk GMM
n,,,,,
200,1.0,0.274,0.050,0.395,0.419
600,1.0,0.283,0.059,0.398,0.417
1000,1.0,0.261,0.048,0.351,0.382
1400,1.0,0.234,0.061,0.349,0.393
1800,1.0,0.265,0.060,0.391,0.405


## Concise results check

The following checks use $\kappa=1$ and $n=1800$. They compare local-risk GMM with equal GMM and the best single channel over each prespecified local-deviation set. The OLS risk row is empirical only. Full boundary-point decompositions, coverage, and interval lengths remain in the CSV summaries.


In [3]:
for regime in regimes:
    panel = worst[(worst["regime"] == regime) & (worst["kappa"] == 1.0) & (worst["n"] == 1800)]
    values = panel.set_index("method")
    local = values.loc["local_risk"]
    equal = values.loc["equal"]
    best_single = min(values.loc["iv", "worst_scaled_mse"], values.loc["sv", "worst_scaled_mse"])
    improvement = 100 * (1 - local["worst_scaled_mse"] / equal["worst_scaled_mse"])
    print(f"{regime_labels[regime]}: local-risk worst-case n*MSE={local['worst_scaled_mse']:.3f}; "
          f"equal={equal['worst_scaled_mse']:.3f}; best single={best_single:.3f}; "
          f"change vs equal={improvement:+.1f}%")
    print(f"  population minimax risk={local['theoretical_worst_risk']:.3f}; "
          f"mean IV information share={local['mean_iv_share']:.3f}; "
          f"minimum robust coverage={local['min_robust_coverage']:.3f}")

valid = worst[(worst["regime"] == "none") & (worst["kappa"] == 0)]
reference = valid.pivot(index="n", columns="method", values="worst_scaled_mse")
print("At kappa=0, largest absolute empirical risk difference between local-risk and equal:",
      round((reference["local_risk"] - reference["equal"]).abs().max(), 6))


IV channel may deviate: local-risk worst-case n*MSE=0.554; equal=0.858; best single=0.933; change vs equal=+35.4%
  population minimax risk=0.556; mean IV information share=0.445; minimum robust coverage=0.959
SV channel may deviate: local-risk worst-case n*MSE=0.753; equal=0.818; best single=1.122; change vs equal=+8.0%
  population minimax risk=0.667; mean IV information share=0.667; minimum robust coverage=0.944
Both channels may deviate: local-risk worst-case n*MSE=0.743; equal=0.839; best single=1.187; change vs equal=+11.5%
  population minimax risk=0.769; mean IV information share=0.617; minimum robust coverage=0.950
At kappa=0, largest absolute empirical risk difference between local-risk and equal: 0.174323
